# Simple: Clean Invalid Values with PAMOLA.CORE

**Goal**: Learn data cleaning basics in 5 minutes using operation.execute()

**What you'll learn:**
- Load sample data from examples/data_examples/
- Apply constraint-based validation using execute()
- Compare before/after results
- Understand data quality improvements

**Prerequisites:**
- Python 3.8+
- PAMOLA.CORE installed
- Basic pandas knowledge

---

## Step 1: Setup and Imports

**How to use:**
- Run this cell to import required libraries
- Auto-detects PAMOLA installation path (searches up 6 levels)
- If not found locally, auto-installs from GitHub

**What you'll see:**
- Detection message showing PAMOLA location or installation progress
- Import confirmation (✅ All imports successful!)
- Notebook start timestamp
- Project root and working directory paths

**Note:** First run may take 1-2 minutes if installing from GitHub

**Expected structure:**
```
PAMOLA/
├── pamola_core/          ← Auto-detected
└── examples/
    ├── data_examples/    ← Data directory
    └── transformations/
        └── 01_clean_invalid_values_simple.ipynb  ← You are here
```

In [ ]:
import pandas as pd
import numpy as np
import sys
import os
import json
from pathlib import Path
from datetime import datetime
from IPython.display import Image, display
 
print("🔍 Detecting PAMOLA installation...\n") 
 
# Auto-detect project root (search up 6 levels for pamola_core/) 
current_dir = Path.cwd() 
project_root = current_dir 
pamola_found = False

for level in range(6): 
    if (project_root / 'pamola_core').exists(): 
        print(f"✓ Found pamola_core at level {level}: {project_root}") 
        sys.path.insert(0, str(project_root))
        pamola_found = True
        break 
    project_root = project_root.parent 

# If not found locally, try to install from Git
if not pamola_found:
    print("⚠️  pamola_core not found in project structure")
    print("📦 Attempting to install from GitHub...\n")
    
    try:
        import subprocess
        
        # Install from Git repository
        install_cmd = [
            sys.executable, "-m", "pip", "install",
            "git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        ]
        
        result = subprocess.run(
            install_cmd,
            capture_output=True,
            text=True,
            check=True
        )
        
        print("✅ Successfully installed pamola_core from GitHub!")
        print(f"   Revision: pre-epic3")
        
    except subprocess.CalledProcessError as e:
        print(f"❌ Installation failed!")
        print(f"   Error: {e.stderr}")
        raise RuntimeError(
            "Could not find pamola_core locally and installation from GitHub failed. "
            "Please install manually using:\n"
            "pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3"
        )
    except Exception as e:
        print(f"❌ Unexpected error during installation: {e}")
        raise

# Now try to import the required modules
try:
    from pamola_core.transformations.cleaning.clean_invalid_values import (
        CleanInvalidValuesOperation
    )
    from pamola_core.utils.ops.op_data_source import DataSource
    from pamola_core.utils.progress import HierarchicalProgressTracker
    from pamola_core.utils.tasks.task_reporting import TaskReporter
    from pamola_core.io.csv import read_csv
    
    print("✅ All imports successful!")
    print(f"📅 Notebook started: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print("=" * 80)
    print(f"📂 Project root:  {project_root.name}")
    print(f"📂 Working dir:   {current_dir.relative_to(project_root)}")
    print("=" * 80)
    
except ImportError as e:
    print(f"❌ Import failed: {e}")
    print("\n💡 Troubleshooting:")
    print("   1. Try restarting your Python kernel/notebook")
    print("   2. Verify installation: pip list | grep pamola")
    print("   3. Manual install: pip install git+https://github.com/DGT-Network/PAMOLA.git@pre-epic3")
    raise

## Step 2: Load Sample Data

**How to use:**
- Run to load data from `examples/data_examples/sample.csv`
- Auto-creates sample data if file missing
- Review dataset structure before proceeding

**What you'll see:**
- Dataset summary (20 records, 5 columns)
- First 5 records preview
- Column statistics with types and ranges
- Fields with invalid values: years_of_experience (negatives, outliers), score (out of range), email (bad formats)

**Note:** Sample includes intentional invalid values for demonstration (negative years, invalid emails, out-of-range scores)

In [ ]:
examples_dir = project_root / 'examples'
data_path = examples_dir / 'data_examples' / 'sample.csv'

if not data_path.exists():
    print("⚠️  File not found, creating sample data...")
    data_path.parent.mkdir(parents=True, exist_ok=True)
    
    # Create sample data with various types and invalid values
    sample_data = pd.DataFrame({
        'record_id': range(1, 21),
        'years_of_experience': [25, 30, -5, 45, 150, 28, 35, 200, 42, 38, 29, 55, -10, 40, 33, 27, 48, 31, 36, 175],
        'score': [85.5, 92.3, 78.9, 110.5, 88.7, 95.2, 82.1, -5.3, 90.8, 87.4, 93.6, 79.2, 86.5, 120.0, 91.3, 84.7, 89.1, 94.5, 81.8, -15.2],
        'category': ['A', 'B', 'C', 'D', 'A', 'Invalid', 'B', 'C', 'E', 'A', 'B', 'Unknown', 'C', 'D', 'A', 'B', 'C', 'BadValue', 'A', 'B'],
        'date': ['2023-01-15', '2023-02-20', '2023-13-05', '2023-03-10', '2023-04-25', '2023-05-30', '2023-06-15', '2025-12-31', '2023-08-10', '2023-09-05', '2023-10-20', '2020-01-01', '2023-11-15', '2030-06-30', '2023-12-25', '2023-01-30', '2023-02-14', '1990-05-20', '2023-03-08', '2023-04-18'],
        'email': ['valid@test.com', 'user@example.com', 'invalid-email', 'test@site.org', 'badformat', 'good@domain.com', 'user@test.com', 'noemail', 'valid@example.org', 'test@domain.com', 'user@site.com', 'invalid@', 'good@test.org', '@nodomain.com', 'user@example.com', 'test@site.com', 'valid@domain.org', 'bad email', 'user@test.org', 'test@example.com']
    })
    sample_data.to_csv(data_path, index=False)
    print(f"✓ Sample data created")

df = read_csv(data_path)
print(f"📊 Dataset loaded: {len(df)} records, {len(df.columns)} columns")
print(f"\n🔍 First 5 records:")
print(df.head())

print(f"\n📈 Column Statistics:")
for col in df.columns:
    dtype_str = str(df[col].dtype)
    if pd.api.types.is_string_dtype(df[col]):
        print(f"  {col:15s} ({dtype_str:10s}): {df[col].nunique()} unique values")
    else:
        print(f"  {col:15s} ({dtype_str:10s}): min={df[col].min()}, max={df[col].max()}")

## Step 3: Setup Task Environment

**How to use:**
1. **CUSTOMIZE field_constraints** in `create_config_kwargs()`
   - Define validation rules for YOUR dataset's fields
   - Set constraint types: valid_range, valid_pattern, allowed_values, date_range
2. Run to validate configuration and setup environment

**What you'll see:**
- Fields to validate with constraint types listed
- Task directory created (✓)
- TaskReporter initialized (✓)
- DataSource and progress tracker ready (✓)

**Note:** Constraints define rules for data validation - invalid values will be nullified or replaced

In [ ]:
# Configuration helper function (production pattern)
def create_config_kwargs():
    return {
        "dataset_name": "main_dataset",
        "field_constraints": {
            "years_of_experience": {
                "data_type": "numeric",
                "constraint_type": "valid_range",
                "min_value": 1,
                "max_value": 15
            },
            "email": {
                "data_type": "text",
                "constraint_type": "valid_pattern",
                "valid_pattern": r'^[\w\.-]+@[\w\.-]+\.\w+$'
            }
        }
    }

kwargs = create_config_kwargs()
field_constraints = kwargs["field_constraints"]

# Validate fields exist
print(f"\n🔍 Validating field configuration...\n")
for field_name in field_constraints.keys():
    if field_name not in df.columns:
        raise ValueError(
            f"❌ Field '{field_name}' not found in dataset!\n"
            f"Available columns: {', '.join(df.columns)}\n"
            f"Please update field_constraints in create_config_kwargs()"
        )

print(f"✓ Fields to validate: {', '.join(field_constraints.keys())}")
for field, config in field_constraints.items():
    print(f"  {field}: {config['constraint_type']}")

# Create task directory
task_dir = examples_dir / 'data_examples' / 'simple_output'
os.makedirs(task_dir, exist_ok=True)
print(f"\n✓ Task directory: {task_dir}")

# Initialize TaskReporter
task_reporter = TaskReporter(
    task_id="simple_001",
    task_type="data_cleaning",
    description="Clean invalid values using constraint-based validation.",
    report_path=task_dir
)
print("✓ TaskReporter initialized")

execute_kwargs = {"dataset_name": kwargs["dataset_name"]}
print(f"✓ Config kwargs: {execute_kwargs}")

# Create DataSource
data_source = DataSource(dataframes={"main_dataset": df})
print("✓ DataSource created")

# Setup progress tracker
tracker = HierarchicalProgressTracker(
    total=6,
    description="Processing validation fields",
    unit="steps",
    track_memory=True,
    level=0,
    bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]"
)
print("✓ Progress tracker ready")

print(f"\n✅ Environment setup complete!")

## Step 4: Configure & Execute Operation

**How to use:**
- Review configuration parameters below
- Run to execute constraint-based validation
- Monitor progress bar (6 steps: load → validate → check constraints → nullify → save → complete)

**Key parameters:**
- `field_constraints`: Validation rules per field
- `mode='REPLACE'`: Modify original fields in-place
- `null_replacement=None`: Keep as null (or use 'mean', 'median', 'mode')
- Constraint types: valid_range, valid_pattern, allowed_values, date_range

**What you'll see:**
- Configuration summary with constraint count
- Progress bar: load → validate fields → apply constraints → replace invalid → save → complete
- Completion status: "completed" (verify no errors)
- Artifact count (3-5 files expected)

**Note:** Invalid values are replaced with null (or strategy value) - original data quality improved

In [ ]:
# Create operation with production-style configuration
operation = CleanInvalidValuesOperation(
    # Core parameters
    field_constraints=field_constraints,
    mode='REPLACE',
    
    # Output configuration
    column_prefix='_',
    output_field_name=None,
    output_format='csv',
    
    # Validation sources
    whitelist_path=None,
    blacklist_path=None,
    
    # Null replacement strategy
    null_replacement=None,  # Keep as null, or use 'mean', 'median', 'mode', etc.
    
    # Processing settings
    use_vectorization=False,
    parallel_processes=1,
    use_cache=False,
    
    # Execution behavior
    generate_visualization=True,
    save_output=True,
    force_recalculation=False
)

print("✓ Operation configured")
print(f"  Fields validated:  {len(field_constraints)}")
print(f"  Mode:              {operation.mode}")
print(f"  Null replacement:  {operation.null_replacement or 'None (keep null)'}")
print(f"  Visualizations:    {operation.generate_visualization}")
print(f"  Save output:       {operation.save_output}")
print(f"  Force recalc:      {operation.force_recalculation}")

# Execute operation
print("\n" + "=" * 80)
print("⚙️  Executing clean invalid values operation...")
print("=" * 80 + "\n")

result = operation.execute(
    data_source=data_source,
    task_dir=task_dir,
    reporter=task_reporter,
    progress_tracker=tracker,
    **execute_kwargs
)

print("\n" + "=" * 80)
print(f"✅ Operation completed!")
print(f"   Status: {result.status}")
print(f"   Artifacts: {len(result.artifacts)}")
print("=" * 80)

## Step 5: Analyze Results

**How to use:**
- Run to load processed data from task_dir
- Review invalid value counts by field
- Compare original vs cleaned data

**What you'll see:**
- Sample processed records (invalid values now null)
- Invalid values by field (count and percentage)
- Visual bar chart showing violation rates
- Summary: original vs processed record counts
- Key metrics: execution time, records processed, validation stats

**Note:** Higher invalid counts indicate data quality issues - clean data improves downstream analysis

In [ ]:
# Display generated artifacts
if result.artifacts:
    print("\n📦 Generated Artifacts:")
    print("=" * 80)
    for artifact in result.artifacts:
        print(f"   [{artifact.category}] {artifact.artifact_type}: {artifact.path.name}")

# Find and load output file
output_files = list(task_dir.glob('output/*.csv'))
if output_files:
    output_file = output_files[0]
    result_df = pd.read_csv(output_file)
    
    print("\n" + "=" * 80)
    print("📊 RESULTS COMPARISON")
    print("=" * 80)
    
    # Show sample records
    print("\n🔍 Sample Processed Records:")
    display_cols = list(field_constraints.keys())
    print(result_df[display_cols].head(10))
    
    # Field-by-field comparison
    print(f"\n📈 Invalid Values by Field:")
    for field in field_constraints.keys():
        original_nulls = df[field].isna().sum()
        processed_nulls = result_df[field].isna().sum()
        invalid_count = processed_nulls - original_nulls
        
        if invalid_count > 0:
            pct = (invalid_count / len(df)) * 100
            bar = '█' * int(pct / 2)
            print(f"  {field:15s} {bar:20s} {invalid_count:2d} invalid ({pct:5.1f}%)")
    
    print("\n" + "=" * 80)
    print("✨ SUMMARY")
    print("=" * 80)
    print(f"  Original records:   {len(df)}")
    print(f"  Processed records:  {len(result_df)}")
    
    # Display result metrics
    if result.metrics:
        print("\n📊 Key Metrics:")
        for key, value in list(result.metrics.items())[:10]:
            if isinstance(value, (int, float)):
                if isinstance(value, float):
                    print(f"  {key:30s}: {value:.4f}")
                else:
                    print(f"  {key:30s}: {value}")
    
    print(f"\n💡 Invalid values nullified for better data quality!")
else:
    print("⚠️  No output file found in", task_dir / 'output')

## Step 6: Review Artifacts Location

**How to use:**
- Run to display all generated files
- Navigate to directories for manual inspection
- Verify each folder has expected file count

**What you'll see:**
- Directory structure tree (output/, metrics/, visualizations/, config/)
- File count per directory (typically 1-2 files each)
- File names with sizes in KB
- Absolute path to task directory for manual navigation

**Output structure:**
```
examples/data_examples/simple_output/
├── output/           # Cleaned CSV (invalid values nullified)
├── metrics/          # Validation metrics JSON
├── visualizations/   # Violation charts (PNG)
└── config/           # Operation configuration
```

In [ ]:
print("\n📂 OUTPUT DIRECTORY STRUCTURE:")
print("=" * 80)
print(f"\nTask directory: {task_dir.absolute()}\n")

for subdir in ['output', 'metrics', 'visualizations', 'config']:
    path = task_dir / subdir
    if path.exists():
        files = list(path.glob('*'))
        print(f"📁 {subdir}/ ({len(files)} files)")
        for f in files[:5]:
            size_kb = f.stat().st_size / 1024
            print(f"   • {f.name} ({size_kb:.1f} KB)")

print("\n✅ All artifacts saved successfully!")

## Step 7: Detailed Artifact Review

**How to use:**
- Review all generated artifacts in detail
- Automatically loads the NEWEST files from each category
- Excludes system files (like data_types metrics)
- Displays visualizations inline for easy review

**What you'll see:**
1. **Metrics JSON**: Operation performance and effectiveness metrics
2. **Output Data**: Processed records with sample rows
3. **Visualizations**: Charts and graphs from the latest operation run

**Note:** Only the most recent files are shown to avoid confusion from multiple runs

In [ ]:
print("\n📊 DETAILED ARTIFACT REVIEW")
print("=" * 80)

# 1. METRICS JSON (NEWEST - with filtering)
print("\n1️⃣ METRICS (JSON):")
print("-" * 80)

metrics_dir = task_dir / 'metrics'
if not metrics_dir.exists():
    print(f"⚠️  Metrics directory not found: {metrics_dir}")
else:
    metrics_files = list(metrics_dir.glob('*.json'))
    
    if not metrics_files:
        print("⚠️  No metrics files found")
    else:
        # 1) Exclude data_types files
        filtered = [f for f in metrics_files if not f.name.startswith("data_types")]

        if filtered:
            # Use only filtered files
            target_files = filtered
            print(f"✓ Found {len(filtered)} filtered metrics file(s) (excluded data_types)")
        else:
            # Fallback: nothing left after filtering → use all
            target_files = metrics_files
            print(f"⚠ No filtered metrics found → fallback to ALL {len(metrics_files)} file(s)")

        # 2) Pick latest
        target_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_metrics_file = target_files[0]
        
        print(f"📄 Loading LATEST: {latest_metrics_file.name}")
        mtime = datetime.fromtimestamp(latest_metrics_file.stat().st_mtime)
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_metrics_file.stat().st_size / 1024:.1f} KB\n")
        print("=" * 80)
        
        try:
            with open(latest_metrics_file, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Extract metadata and metrics
            metadata = data.get('metadata', {})
            metrics = data.get('metrics', {})
            
            # -------- METADATA --------
            print("📋 METADATA:")
            print(f"   Timestamp: {metadata.get('timestamp', 'N/A')}")
            print(f"   Name: {metadata.get('name', 'N/A')}")

            op = metadata.get("operation", {})
            print(
                f"   Operation: {op.get('class', 'N/A')}."
                f"{op.get('function', 'N/A')}"
            )

            # -------- INVALID VALUES --------
            invalid_vals = metrics.get("invalid_values", {})
            if invalid_vals:
                print("\n📊 INVALID VALUES BY FIELD:")
                print("-" * 80)

                for field, stats in invalid_vals.items():
                    print(f"   🔸 Field: {field}")
                    print(f"      Count:           {stats.get('count', 0)}")
                    print(f"      Percent:         {stats.get('percent', 0):.2%}")
                    print(f"      Constraint Type: {stats.get('constraint_type', 'N/A')}")
                    print(f"      Whitelist:       {'Yes' if stats.get('whitelist') else 'No'}")
                    print(f"      Blacklist:       {'Yes' if stats.get('blacklist') else 'No'}")
            else:
                print("\n📊 INVALID VALUES BY FIELD:")
                print("   (No invalid values recorded)")

            # -------- CONSTRAINT TYPE VIOLATIONS --------
            violations = metrics.get("constraint_type_violations", {})
            if violations:
                print("\n📈 CONSTRAINT TYPE VIOLATIONS:")
                print("-" * 80)
                for constraint_type, count in violations.items():
                    print(f"   {constraint_type:30s}: {count}")
            else:
                print("\n📈 CONSTRAINT TYPE VIOLATIONS:")
                print("   (None)")

            # -------- TOP VIOLATIONS --------
            top_violations = metrics.get("top_violations", [])
            if top_violations:
                print("\n🔝 TOP VIOLATIONS:")
                print("-" * 80)
                for i, v in enumerate(top_violations, 1):
                    print(f"   {i}. {v}")
            else:
                print("\n🔝 TOP VIOLATIONS:")
                print("   (None)")

            # -------- PERFORMANCE --------
            print("\n⚡ PERFORMANCE:")
            print("-" * 80)

            exec_time = metrics.get("execution_time_seconds")
            if exec_time is not None:
                print(f"   Execution Time:     {exec_time:.4f}s")

            records = metrics.get("records_processed")
            if records is not None:
                print(f"   Records Processed:  {records:,}")

            rps = metrics.get("records_per_second")
            if rps is not None:
                print(f"   Records / Second:   {rps:.2f}")
            
        except json.JSONDecodeError as e:
            print(f"❌ JSON decode error: {e}")
        except Exception as e:
            print(f"❌ Error reading metrics: {e}")

# 2. OUTPUT DATA (NEWEST)
print("\n\n2️⃣ OUTPUT DATA (First 10 rows):")
print("-" * 80)

output_dir = task_dir / 'output'
if not output_dir.exists():
    print(f"⚠️  Output directory not found: {output_dir}")
else:
    output_files = list(output_dir.glob('*.csv'))
    
    if not output_files:
        print("⚠️  No output files found")
    else:
        # Sort by modification time (newest first)
        output_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        latest_output_file = output_files[0]
        
        # Show file info
        mtime = datetime.fromtimestamp(latest_output_file.stat().st_mtime)
        print(f"✓ Found {len(output_files)} output file(s)")
        print(f"📄 Loading LATEST: {latest_output_file.name}")
        print(f"   Modified: {mtime.strftime('%Y-%m-%d %H:%M:%S')}")
        print(f"   Size: {latest_output_file.stat().st_size / 1024:.1f} KB\n")
        
        try:
            output_df = pd.read_csv(latest_output_file)
            print(f"📊 Shape: {output_df.shape[0]} rows × {output_df.shape[1]} columns")
            print(f"\n{output_df.head(10).to_string()}")
            
            # Show null counts by field
            print("\n\n📊 Null Counts by Field:")
            print("-" * 80)
            for field in field_constraints.keys():
                if field in output_df.columns:
                    null_count = output_df[field].isna().sum()
                    null_pct = (null_count / len(output_df)) * 100
                    print(f"   {field:15s}: {null_count:3d} nulls ({null_pct:5.1f}%)")
            
        except Exception as e:
            print(f"❌ Error reading output: {e}")

# 3. VISUALIZATIONS (NEWEST BATCH)
print("\n\n3️⃣ VISUALIZATIONS:")
print("-" * 80)

viz_dir = task_dir / 'visualizations'
if not viz_dir.exists():
    print(f"⚠️  Visualizations directory not found: {viz_dir}")
else:
    viz_files = list(viz_dir.glob('*.png'))
    
    if not viz_files:
        print("⚠️  No visualizations found")
    else:
        # Sort by modification time (newest first)
        viz_files.sort(key=lambda x: x.stat().st_mtime, reverse=True)
        
        # Get timestamp from latest file to identify the batch
        if viz_files:
            latest_time = viz_files[0].stat().st_mtime
            # Group files from same operation (within 10 seconds)
            time_threshold = 10  # seconds
            latest_viz_batch = [
                f for f in viz_files 
                if abs(f.stat().st_mtime - latest_time) < time_threshold
            ]
            
            # Sort batch by name for consistent ordering
            latest_viz_batch.sort(key=lambda x: x.name)
            
            # Show info
            latest_mtime = datetime.fromtimestamp(latest_time)
            print(f"✓ Found {len(viz_files)} total visualization(s)")
            print(f"📄 Showing LATEST batch: {len(latest_viz_batch)} files")
            print(f"   Created: {latest_mtime.strftime('%Y-%m-%d %H:%M:%S')}\n")
            
            # Display each visualization from latest batch
            for i, viz_file in enumerate(latest_viz_batch, 1):
                # Extract readable name from filename
                viz_name = viz_file.stem.replace('_', ' ').title()
                
                print(f"\n{i}. {viz_name}")
                print(f"   File: {viz_file.name}")
                print("-" * 60)
                try:
                    display(Image(filename=str(viz_file)))
                except Exception as e:
                    print(f"   ⚠️  Could not display: {e}")
        
        # Show info about older visualizations if any
        if len(viz_files) > len(latest_viz_batch):
            print(f"\n💡 Note: {len(viz_files) - len(latest_viz_batch)} older visualization(s) not shown")

print("\n\n" + "=" * 80)
print("✅ ARTIFACT REVIEW COMPLETE")
print("=" * 80)

## 🎯 Summary

**What you learned:**

✅ Load data from examples/data_examples/  
✅ Setup environment with TaskReporter, DataSource, ProgressTracker  
✅ Configure clean invalid values with full parameters  
✅ Execute using operation.execute()  
✅ Analyze results and review artifacts  

**Key takeaways:**
- Constraint-based validation identifies invalid values
- Multiple constraint types: numeric ranges, allowed values, date ranges, patterns
- Invalid values are nullified (or replaced based on strategy)
- Whitelist/blacklist support for external validation
- Comprehensive metrics show data quality improvements

## 📚 Next Steps

**Ready for advanced features?** Try:

📘 **`02_clean_invalid_values_advanced.ipynb`** includes:
- Custom validation functions
- Advanced null replacement strategies
- Multi-field constraint validation
- Batch processing for large datasets
- Integration with data quality workflows

**Resources:**
- 📖 [PAMOLA Documentation](../../docs/)
- 🔧 [API Reference](../../docs/api/)